# 01 · Raw data profile
**Project:** Sales Variance FP&A Dashboard

This notebook only **looks at** the raw file `data/raw/sales_raw_messy.xlsx` to show why cleaning is needed. It changes nothing.

* The raw file is **synthetic**. It is made by `scripts/generate_synthetic_data.py` (fixed seed, so everyone gets the same file).
* The cleaning is done by `scripts/clean_sales_data.py` (notebook 02 checks its result).
* The generator also writes `data/raw/injected_defects.csv`, the answer key of every defect it planted. The cleaner never reads it; it is only used to *test* the cleaner.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Works whether Jupyter starts in notebooks/ or in the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from fpa_lib import config  # noqa: E402

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 40
pd.options.display.max_colwidth = 90
rel = lambda p: p.relative_to(ROOT).as_posix()  # noqa: E731  (print paths without your local folder names)
from fpa_lib import cleaning as cl  # noqa: E402

## 1. Load the raw file
Everything is read as text/objects (`dtype=object`) and `keep_default_na=False`, so pandas does not silently turn the text `NA` (North America) into a missing value or change any number.

In [2]:
raw = pd.read_excel(config.RAW_FILE, dtype=object, keep_default_na=False, na_values=[])
print(f"File: {rel(config.RAW_FILE)}   rows: {len(raw):,}   columns: {raw.shape[1]}")
padded = [c for c in raw.columns if c != c.strip()]
extra = [c.strip() for c in raw.columns if c.strip() not in config.CLEAN_COLUMNS]
print(f"Headers with extra spaces: {padded}")
print(f"Columns not in the schema: {extra}")
raw.columns = [c.strip() for c in raw.columns]     # same header trim the cleaner does, so the rest can use names
raw.head()

File: data/raw/sales_raw_messy.xlsx   rows: 21,815   columns: 20
Headers with extra spaces: [' Actual_Qty', 'Target_Sales_USD ']
Columns not in the schema: ['Comments']


,Line_ID,Date,Year,Month,Month_Name,Region,Category,Product,Target_Qty,Actual_Qty,Target_Unit_Price_USD,Actual_Unit_Price_USD,Target_Unit_Cost_USD,Actual_Unit_Cost_USD,Target_Sales_USD,Actual_Sales_USD,Target_Gross_Profit_USD,Actual_Gross_Profit_USD,Notes,Comments
0,APA-HOM02-202401-005,2024-01-01 00:00:00,2024,1,January,Asia-Pacific,Home Appliances,air purifier,132,110,183.33,185.21,121.34,123.30,"24,199.56","20,373.10","8,182.68","6,810.10",,
1,APA-HOM02-202401-004,2024-01-01 00:00:00,2024,1,January,Asia Pacific,Home Appliances,Air Purifier,99,106,183.33,183.10,121.34,120.32,"18,149.67",19408.60,"6,137.01","6,654.68",,
2,APA-HOM01-202401-005,2024-01-01 00:00:00,2024,1,January,AsiaPac,Home Appliances​,Air Fryer,171,146,117.74,117.63,77.23,77.56,"20,133.54","17,173.98","6,927.21","5,850.22",,
3,APA-HOM02-202401-003,2024-01-01 00:00:00,2024,1,January,Asia Pacific,Home Appliances,Air Purifier,53,45,177.83,178.93,121.34,121.14,"9,424.99","8,051.85","2,993.97","2,600.55",,
4,APA-HOM02-202401-002,2024-01-01 00:00:00,2024,1,January,Asia Pacific,Home Appliances,Air Purifier,83,72,187,189.48,121.34,121.78,15521,"13,642.56","5,449.78","4,874.40",,


## 2. Blank cells and 'missing' spellings
A cell counts as missing if it is empty or a text such as `NULL`, `N/A`, `-` (but `NA` is a real value in the Region column).

In [3]:
miss = pd.DataFrame({
    "empty": raw.apply(lambda s: s.map(lambda v: cl.clean_text(v) is None).sum()),
    "null_spelling": raw.apply(lambda s: s.map(lambda v: cl.clean_text(v) is not None and cl.is_null(v, s.name)).sum()),
})
miss["missing_pct"] = (miss.sum(axis=1) / len(raw) * 100).round(2)
miss[miss.sum(axis=1) > 0].sort_values("missing_pct", ascending=False)

,empty,null_spelling,missing_pct
Comments,21815,0,100.00
Notes,18931,0,86.78
Region,671,0,3.08
Actual_Qty,148,448,2.73
Category,460,0,2.11
Actual_Unit_Cost_USD,171,290,2.11
Actual_Gross_Profit_USD,100,269,1.69
Actual_Unit_Price_USD,79,269,1.60
Target_Gross_Profit_USD,150,181,1.52
Target_Unit_Price_USD,136,167,1.39


## 3. Numbers stored as text, and unreadable numbers

In [4]:
num_cols = config.QTY_COLUMNS + config.MONEY_COLUMNS
status = {c: raw[c].map(lambda v, c=c: (cl.parse_qty if c.endswith("_Qty") else cl.parse_money)(v)[1]).value_counts()
          for c in num_cols}
pd.DataFrame(status).T.fillna(0).astype(int)

,ok,text,missing
Target_Qty,20532,1051,232
Actual_Qty,20179,1040,596
Target_Unit_Price_USD,20446,1066,303
Actual_Unit_Price_USD,20445,1022,348
Target_Unit_Cost_USD,20484,1098,233
Actual_Unit_Cost_USD,20302,1052,461
Target_Sales_USD,20526,1091,198
Actual_Sales_USD,20437,1112,266
Target_Gross_Profit_USD,20415,1069,331
Actual_Gross_Profit_USD,20394,1052,369


`text` = a valid number typed as text (e.g. `$1,234.50`), `invalid` = cannot be read as an amount with at most 2 decimals. Invalid values are treated as missing and recovered only if an identity gives the exact answer.

## 4. Dates in several formats

In [5]:
raw["Date"].map(lambda v: cl.date_candidates(v)[1]).value_counts().to_frame("rows")

,rows
Date,
date,15248
text,5697
excel serial,623
missing,247


## 5. Inconsistent labels

In [6]:
region = raw["Region"].map(cl.clean_text).fillna("(blank)")
print(f"Distinct Region spellings: {region.nunique()} (the clean data has {len(config.REGIONS)})")
region.value_counts().to_frame("rows")

Distinct Region spellings: 32 (the clean data has 5)


,rows
Region,
North America,2740
Europe,2701
Asia Pacific,2609
Middle East,1626
Latin America,1584
(blank),671
AsiaPac,478
asia pacific,468
APAC,461


In [7]:
prod = raw["Product"].map(cl.clean_text).dropna()
print(f"Distinct Product spellings: {prod.nunique()}  ->  products in the master: {len(config.PRODUCTS)}")
print(f"Spellings not in the explicit mapping table: {(prod.map(lambda p: cl.map_label(p, cl.PRODUCT_ALIASES)[0]).isna()).sum():,}")

Distinct Product spellings: 81  ->  products in the master: 27
Spellings not in the explicit mapping table: 0


## 6. Duplicates
* **Exact duplicates**: every cell identical.
* **Repeated Line_ID**: the record key appears more than once (copies may complement each other or conflict).

In [8]:
blank = raw.map(cl.is_null).all(axis=1)
dup = raw[~blank].astype(str).duplicated()          # identical raw content in every column (as the cleaner tests it)
print(f"Entirely blank rows ............. {blank.sum():,}")
print(f"Exact duplicate rows ............ {dup.sum():,}")
ids = raw.loc[~blank, "Line_ID"].map(cl.clean_text)
print(f"Rows without a Line_ID .......... {ids.isna().sum():,}")
print(f"Extra copies of a Line_ID (after removing exact duplicates): {ids[~dup].dropna().duplicated().sum():,}")

Entirely blank rows ............. 25
Exact duplicate rows ............ 525
Rows without a Line_ID .......... 6
Extra copies of a Line_ID (after removing exact duplicates): 272


## 7. Rows that break the arithmetic
For a correct line, `Sales = Qty x Unit_Price` and `Gross_Profit = Sales - Qty x Unit_Cost` hold **exactly** (in cents). Count the rows where every value is readable but an identity fails.

In [9]:
def broken(side):
    f = cl.SIDES[side]
    parsed = pd.DataFrame({k: raw[c].map(lambda v, c=c: (cl.parse_qty if c.endswith("_Qty") else cl.parse_money)(v)[0])
                           for k, c in f.items()})
    full = parsed.notna().all(axis=1)
    ok = parsed[full].apply(lambda r: cl.identities_hold({k: int(r[k]) for k in f}), axis=1)
    return int(full.sum()), int((~ok).sum())

for side in cl.SIDES:
    n, bad = broken(side)
    print(f"{side:7}: {n:,} fully readable rows, {bad:,} break an identity")

Target : 20,618 fully readable rows, 85 break an identity


Actual : 19,918 fully readable rows, 372 break an identity


## 8. The answer key (for testing only)
What the generator says it injected. The test suite checks that the cleaner fixes each defect exactly, or sends the row to review.

In [10]:
manifest = pd.read_csv(config.DEFECT_MANIFEST_FILE, dtype=str, keep_default_na=False)
manifest.groupby(["Defect", "Expected_Handling"]).size().rename("cells").to_frame()

cells
Defect                            Expected_Handling                              
Conflicting duplicate             Review (conflict) - all copies               20
Date conflicts with Line_ID       Review (conflict)                             8
Exact duplicate                   Removed (exact duplicate)                   525
Label conflicts with Line_ID      Review (conflict)                            20
Logical duplicate (complementary) Merged into first copy                      252
Missing Line_ID                   Review (no key)                               6
Missing value                     Recovered (exact)                          2666
                                  Recovered from Line_ID                      840
                                  Recovered from product master               420
Missing value (two linked fields) Review (unrecoverable)                       80
Sign error                        Recovered (exact)                           126
Transposed digits                 Recovered (exact) or review if ambiguous     83
Typo (x10)                        Recovered (exact)                           105
Wrong derived value               Recomputed from Date                        209
                                  Recovered (exact) or review if ambiguous     84
Zero price                        Recovered (exact)                            42

**Next:** notebook 02 shows what the cleaning script did about each of these problems.